In [1]:
import pandas as pd
import os

# --- Configuration ---
file_path = "master_dataset.parquet"
# ---

# IMPORTANT: To save memory, we will only load the single column we need for this question.
print(f"Loading the 'region' column from {file_path}...")
df_region = pd.read_parquet(file_path, columns=['region'])

print("\n--- Games per Region ---")
region_counts = df_region['region'].value_counts()
print(region_counts)

Loading the 'region' column from master_dataset.parquet...

--- Games per Region ---
region
EUW1    15815347
KR      12407741
NA1      7059084
OC1       936110
Name: count, dtype: int64


In [2]:
import pandas as pd

# --- Configuration ---
file_path = "master_dataset.parquet"
# ---

# 1. Load ONLY the two columns we need for this question.
# This is very memory-efficient.
print(f"Loading 'region' and 'averageTier' columns from {file_path}...")
df_ranks = pd.read_parquet(file_path, columns=['region', 'averageTier'])
print("Loading complete.")

# 2. Group by both region and tier, and then count the number of games.
# The .value_counts() method is the perfect tool for this.
# It automatically finds all unique combinations and counts them.
print("\n--- Games per Rank in Each Region ---")
rank_distribution = df_ranks.value_counts(['region', 'averageTier'])

# 3. Print the result.
# The output will be a table showing the game counts for each rank within each region.
print(rank_distribution)

Loading 'region' and 'averageTier' columns from master_dataset.parquet...
Loading complete.

--- Games per Rank in Each Region ---
region  averageTier
EUW1    EMERALD        5562950
        DIAMOND        3782723
KR      DIAMOND        2629365
        EMERALD        2496020
        SILVER         2131935
EUW1    SILVER         2089349
KR      GOLD           1973484
EUW1    GOLD           1953449
NA1     SILVER         1784146
KR      PLATINUM       1652449
NA1     GOLD           1464231
EUW1    PLATINUM       1447128
KR      MASTER         1340290
NA1     EMERALD         970864
        PLATINUM        960055
        DIAMOND         913008
EUW1    MASTER          861324
NA1     MASTER          830399
OC1     SILVER          295663
        GOLD            177796
        DIAMOND         121687
KR      GRANDMASTER     119416
OC1     PLATINUM        118958
        EMERALD         104111
        MASTER           96048
NA1     GRANDMASTER      90170
EUW1    GRANDMASTER      79707
KR      CHAL

In [3]:
import pandas as pd

# --- Configuration ---
file_path = "master_dataset.parquet"
# ---

# 1. Load the necessary columns, same as before.
print(f"Loading 'region' and 'averageTier' columns...")
df_ranks = pd.read_parquet(file_path, columns=['region', 'averageTier'])
print("Loading complete.")

# 2. Define the correct order for the ranks.
# This ensures our final table is sorted logically, not alphabetically.
rank_order = [
    'CHALLENGER', 'GRANDMASTER', 'MASTER', 
    'DIAMOND', 'EMERALD', 'PLATINUM', 
    'GOLD', 'SILVER', 'BRONZE', 'IRON'
]
# Convert the 'averageTier' column to a special 'Categorical' type with our defined order.
df_ranks['averageTier'] = pd.Categorical(df_ranks['averageTier'], categories=rank_order, ordered=True)


# 3. Create the pivot table.
# This is the key step to transform the data into a grid format.
print("\n--- Cleaned Games per Rank in Each Region ---")
#  - .groupby() groups the data by region and tier.
#  - .size() counts the number of games in each group.
#  - .unstack() pivots the 'averageTier' values into columns.
#  - .fillna(0) replaces any missing combinations (e.g., OC1 IRON) with 0.
#  - .astype(int) converts the numbers to clean integers.
rank_pivot_table = df_ranks.groupby(['region', 'averageTier']).size().unstack(fill_value=0)


# 4. Print the final, clean table.
print(rank_pivot_table)

Loading 'region' and 'averageTier' columns...
Loading complete.

--- Cleaned Games per Rank in Each Region ---


C:\Users\lytten\AppData\Local\Temp\ipykernel_12416\3447620989.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  rank_pivot_table = df_ranks.groupby(['region', 'averageTier']).size().unstack(fill_value=0)


averageTier  CHALLENGER  GRANDMASTER   MASTER  DIAMOND  EMERALD  PLATINUM  \
region                                                                      
EUW1              38717        79707   861324  3782723  5562950   1447128   
KR                64782       119416  1340290  2629365  2496020   1652449   
NA1               46211        90170   830399   913008   970864    960055   
OC1                7429        14418    96048   121687   104111    118958   

averageTier     GOLD   SILVER  BRONZE  IRON  
region                                       
EUW1         1953449  2089349       0     0  
KR           1973484  2131935       0     0  
NA1          1464231  1784146       0     0  
OC1           177796   295663       0     0  


In [1]:
import pandas as pd
import requests

# --- Configuration ---
file_path = "master_dataset.parquet"
# ---

# 1. Fetch official champion data
try:
    print("Fetching latest champion data from Riot...")
    ddragon_url = "https://ddragon.leagueoflegends.com/cdn/14.13.1/data/en_US/champion.json"
    response = requests.get(ddragon_url)
    champion_data = response.json()['data']
    id_to_name = {int(details['key']): name for name, details in champion_data.items()}
    print("Champion data fetched successfully.")
except Exception as e:
    print(f"Could not fetch champion data. Error: {e}")
    id_to_name = {}

# 2. --- THE FIX ---
# Apply the correct manual mapping based on your investigation.
id_to_name[799] = 'Ambessa'
id_to_name[800] = 'Mel'
id_to_name[893] = 'Aurora'
print("Manually patched champion mapping with Ambessa, Mel, and Aurora.")

# 3. Load the required columns
roles = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']
champion_cols = [f'team_{team}_{role}_championId' for team in [100, 200] for role in roles]
cols_to_load = champion_cols + ['team_100_win', 'team_200_win']
print(f"\nLoading required columns from {file_path}...")
df_matches = pd.read_parquet(file_path, columns=cols_to_load)
print("Loading complete.")

# 4. Restructure data from Wide to Long
print("Restructuring data...")
player_data_list = []
for team in [100, 200]:
    for role in roles:
        champ_col = f'team_{team}_{role}_championId'
        win_col = f'team_{team}_win'
        temp_df = df_matches[[champ_col, win_col]].copy()
        temp_df.rename(columns={champ_col: 'championId', win_col: 'win'}, inplace=True)
        player_data_list.append(temp_df)
all_players_df = pd.concat(player_data_list, ignore_index=True)

# 5. Calculate win rates, ensuring we only use IDs we have names for
known_ids = list(id_to_name.keys())
filtered_players_df = all_players_df[all_players_df['championId'].isin(known_ids)]
win_rates_df = filtered_players_df.groupby('championId')['win'].agg(['size', 'sum'])
win_rates_df.rename(columns={'size': 'total_games', 'sum': 'wins'}, inplace=True)
win_rates_df['win_rate'] = (win_rates_df['wins'] / win_rates_df['total_games'])

# 6. Format and display the final table
win_rates_df.sort_values('win_rate', ascending=False, inplace=True)
win_rates_df.index = win_rates_df.index.map(id_to_name)
win_rates_df['win_rate'] = (win_rates_df['win_rate'] * 100).map('{:.2f}%'.format)

print("\n--- Corrected Champion Win Rates ---")
print("\n--- Top 20 Highest Win Rates ---")
print(win_rates_df.head(20))
print("\n--- Top 20 Lowest Win Rates ---")
print(win_rates_df.tail(20))

with pd.option_context('display.max_rows', None):
    print(win_rates_df.sort_index())


Fetching latest champion data from Riot...
Champion data fetched successfully.
Manually patched champion mapping with Ambessa, Mel, and Aurora.

Loading required columns from master_dataset.parquet...
Loading complete.
Restructuring data...

--- Corrected Champion Win Rates ---

--- Top 20 Highest Win Rates ---
              total_games     wins win_rate
championId                                 
Taric              608895   318372   52.29%
Cassiopeia        1201750   627182   52.19%
Nilah              689667   358940   52.05%
Warwick           2386967  1237524   51.85%
KogMaw            1056134   547234   51.81%
Quinn              833370   431442   51.77%
Anivia            1081378   559808   51.77%
RekSai             657696   340206   51.73%
Urgot             1213463   626669   51.64%
Janna             1927241   992639   51.51%
Braum             2350850  1210636   51.50%
Heimerdinger       782297   402687   51.47%
Yorick            1775760   913914   51.47%
Naafiri           1589714  

In [2]:
import pandas as pd
import requests

# --- Configuration ---
file_path = "master_dataset.parquet"
# ---

# 1. Fetch and patch the champion ID to Name mapping, just as before.
try:
    print("Fetching latest champion data from Riot...")
    ddragon_url = "https://ddragon.leagueoflegends.com/cdn/14.13.1/data/en_US/champion.json"
    response = requests.get(ddragon_url)
    champion_data = response.json()['data']
    id_to_name = {int(details['key']): name for name, details in champion_data.items()}
    id_to_name[799] = 'Ambessa'
    id_to_name[800] = 'Mel'
    id_to_name[893] = 'Aurora'
    print("Champion data fetched and patched successfully.")
except Exception as e:
    print(f"Could not fetch champion data. Error: {e}")
    id_to_name = {}

# 2. Define and load the champion ID columns.
roles = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']
champion_cols = [f'team_{team}_{role}_championId' for team in [100, 200] for role in roles]
print(f"\nLoading required champion columns from {file_path}...")
df_champions = pd.read_parquet(file_path, columns=champion_cols)
print("Loading complete.")

# 3. Transform the data from WIDE to LONG.
print("Restructuring data...")
player_roles_list = []
for role in roles:
    for team in [100, 200]:
        champ_col = f'team_{team}_{role}_championId'
        temp_df = pd.DataFrame(df_champions[champ_col])
        temp_df.rename(columns={champ_col: 'championId'}, inplace=True)
        temp_df['role'] = role
        player_roles_list.append(temp_df)
all_player_roles_df = pd.concat(player_roles_list, ignore_index=True)

# 4. Calculate the role distribution percentages.
print("Calculating role distributions...")
# First, filter to only known champion IDs
known_ids = list(id_to_name.keys())
filtered_roles_df = all_player_roles_df[all_player_roles_df['championId'].isin(known_ids)]
role_counts = filtered_roles_df.groupby(['championId', 'role']).size()
role_percentages = role_counts / role_counts.groupby('championId').transform('sum')

# 5. Format and display the final table.
role_dist_table = role_percentages.unstack(fill_value=0)
role_dist_table.index = role_dist_table.index.map(id_to_name)
role_dist_table = role_dist_table.reindex(columns=roles, fill_value=0)
for role in roles:
    role_dist_table[role] = (role_dist_table[role] * 100).map('{:.2f}%'.format)

print("\n--- Corrected Champion Role Distribution ---")
# Display a sample of the data, sorted alphabetically
print(role_dist_table.sort_index().head(25))

with pd.option_context('display.max_rows', None):
    print(role_dist_table.sort_index())


Fetching latest champion data from Riot...
Champion data fetched and patched successfully.

Loading required champion columns from master_dataset.parquet...
Loading complete.
Restructuring data...
Calculating role distributions...

--- Corrected Champion Role Distribution ---
role            TOP  JUNGLE  MIDDLE  BOTTOM UTILITY
championId                                         
Aatrox       96.61%   0.34%   2.61%   0.10%   0.35%
Ahri          1.29%   0.02%  97.26%   0.39%   1.05%
Akali        19.62%   0.02%  80.15%   0.08%   0.14%
Akshan       10.16%   0.17%  86.21%   2.71%   0.75%
Alistar       1.25%   0.08%   0.36%   0.07%  98.24%
Ambessa      82.62%   7.80%   9.03%   0.25%   0.31%
Amumu         0.57%  87.93%   0.12%   0.04%  11.33%
Anivia        6.65%   0.05%  81.24%   1.76%  10.30%
Annie         4.52%   0.08%  80.49%   1.18%  13.74%
Aphelios      0.56%   0.03%   0.67%  98.70%   0.05%
Ashe          0.63%   0.03%   0.36%  91.02%   7.97%
AurelionSol   3.99%   0.87%  89.64%   4.26%   1

In [1]:
import pandas as pd

# --- Configuration ---
file_path = "master_dataset.parquet"
# ---

# 1. Load ONLY the 'matchId' column to be highly memory-efficient.
print(f"Loading the 'matchId' column from {file_path}...")
# This is the most important optimization: we ignore the other 421 columns.
df_matches = pd.read_parquet(file_path, columns=['matchId'])
print("Loading complete.")


# 2. Perform the duplicate check.
# The .duplicated() method returns a boolean (True/False) for each row.
# .sum() then counts all the 'True' values. This is extremely fast.
print("Checking for duplicate matchIDs...")
duplicate_count = df_matches['matchId'].duplicated().sum()


# 3. Report the final result.
print("\n--- Verification Result ---")
if duplicate_count == 0:
    print("✅ Success! No duplicate matchIDs found.")
    print("Each row in the master dataset corresponds to a unique match.")
else:
    print(f"❌ Found {duplicate_count} duplicate matchIDs.")
    print("This indicates that some games are present more than once in the dataset.")

Loading the 'matchId' column from master_dataset.parquet...
Loading complete.
Checking for duplicate matchIDs...

--- Verification Result ---
❌ Found 28633 duplicate matchIDs.
This indicates that some games are present more than once in the dataset.


In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import os
from tqdm.notebook import tqdm # Use the notebook-friendly version of tqdm

# --- Configuration ---
ORIGINAL_FILE = "master_dataset.parquet"
FINAL_FILE = "master_dataset_final.parquet"
BATCH_SIZE = 100000

# --- Data Quality Rules ---
VALID_QUEUE_IDS = [420, 440]
MIN_GAME_DURATION = 600
# ---

def audit_differences_notebook():
    """
    Analyzes the original dataset to provide a breakdown of how many rows
    were removed by each data quality filter.
    """
    if not os.path.exists(ORIGINAL_FILE):
        print(f"❌ Error: The original file '{ORIGINAL_FILE}' could not be found.")
        return
    if not os.path.exists(FINAL_FILE):
        print(f"❌ Error: The final clean file '{FINAL_FILE}' could not be found.")
        return

    # --- Initialize Counters ---
    total_rows_original = 0
    queue_removed = 0
    duration_removed = 0
    invalid_champ_removed = 0
    duplicates_removed = 0
    
    seen_match_ids = set()
    roles = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']
    champion_cols = [f'team_{team}_{role}_championId' for team in [100, 200] for role in roles]

    print(f"--- Starting Audit of '{ORIGINAL_FILE}' ---")
    
    pq_file = pq.ParquetFile(ORIGINAL_FILE)
    batch_iterator = pq_file.iter_batches(batch_size=BATCH_SIZE)
    
    for batch in tqdm(batch_iterator, total=pq_file.num_row_groups, desc="Auditing chunks"):
        chunk = batch.to_pandas()
        total_rows_original += len(chunk)
        
        # --- APPLY FILTERS SEQUENTIALLY AND COUNT ---
        rows_before_queue = len(chunk)
        chunk = chunk[chunk['queueId'].isin(VALID_QUEUE_IDS)]
        queue_removed += rows_before_queue - len(chunk)
        
        rows_before_duration = len(chunk)
        chunk = chunk[chunk['gameDuration'] >= MIN_GAME_DURATION]
        duration_removed += rows_before_duration - len(chunk)
        
        rows_before_champs = len(chunk)
        chunk = chunk[(chunk[champion_cols] > 0).all(axis=1)]
        invalid_champ_removed += rows_before_champs - len(chunk)
        
        rows_before_dupes = len(chunk)
        chunk.drop_duplicates(subset=['matchId'], keep='first', inplace=True)
        is_new_id = ~chunk['matchId'].isin(seen_match_ids)
        new_rows_count = is_new_id.sum()
        duplicates_removed += rows_before_dupes - new_rows_count
        seen_match_ids.update(chunk.loc[is_new_id, 'matchId'])

    # --- Final Report ---
    print("\n--- Data Cleaning Audit Report ---")
    print(f"Original Total Rows: {total_rows_original:,}")
    
    total_removed = queue_removed + duration_removed + invalid_champ_removed + duplicates_removed
    final_rows_calculated = total_rows_original - total_removed
    
    print(f"\nTotal Rows Removed: {total_removed:,}")
    print("Breakdown of Reasons for Removal:")
    if total_removed > 0:
        print(f"  - Invalid Game Mode (non-Ranked): {queue_removed:>12,d} ({queue_removed / total_removed:.1%})")
        print(f"  - Short Game Duration (<10 min):  {duration_removed:>12,d} ({duration_removed / total_removed:.1%})")
        print(f"  - Incomplete/Invalid Drafts:      {invalid_champ_removed:>12,d} ({invalid_champ_removed / total_removed:.1%})")
        print(f"  - Duplicate Matches:              {duplicates_removed:>12,d} ({duplicates_removed / total_removed:.1%})")
    
    print("-" * 40)
    print(f"Calculated Final Rows: {final_rows_calculated:,}")

    final_file_rows = pq.ParquetFile(FINAL_FILE).num_rows
    print(f"Actual Rows in '{FINAL_FILE}': {final_file_rows:,}")
    if final_rows_calculated == final_file_rows:
        print("✅ The calculated total perfectly matches the final file.")
    else:
        print("❌ Warning: The calculated total does NOT match the final file.")

# Run the audit function
audit_differences_notebook()

--- Starting Audit of 'master_dataset.parquet' ---


Auditing chunks:   0%|          | 0/133 [00:00<?, ?it/s]

In [2]:
import pandas as pd
import json
import ast  # <-- IMPORT THE 'ast' MODULE

# --- Configuration ---
# Your paths are correct.
MATCHUP_FILE = "safe/matchup_winrates.json"
MAPPING_FILE = "safe/champion_mapping.json"
# ---

def sanity_check_matchups():
    """
    Loads the matchup data and performs several spot-checks to verify
    its integrity and structure.
    """
    print(f"--- Sanity-Checking '{MATCHUP_FILE}' ---")
    
    # 1. Load the necessary files
    print("Loading data files...")
    with open(MAPPING_FILE, 'r') as f:
        id_to_name = json.load(f)
    
    matchup_df = pd.read_json(MATCHUP_FILE, orient='index')
    
    # --- START OF FIX ---
    # The index is currently a list of strings. We need to convert it to a list of tuples.
    # ast.literal_eval safely evaluates a string containing a Python literal.
    tuples_index = [ast.literal_eval(i) for i in matchup_df.index]
    # --- END OF FIX ---
    
    # Now, we create the MultiIndex from our corrected list of tuples.
    matchup_df.index = pd.MultiIndex.from_tuples(
        tuples_index, # <-- Use the corrected index
        names=['p1_champId', 'p1_role', 'p2_champId', 'p2_role', 'relationship']
    )
    
    print("Data loaded and parsed successfully.")
    print(f"Found {len(matchup_df):,} total matchup entries.")

    # 2. Define a helper function for easy lookups
    def get_matchup_stats(p1_name, p1_role, p2_name, p2_role, relationship):
        name_to_id = {v: k for k, v in id_to_name.items()}
        try:
            p1_id = int(name_to_id[p1_name])
            p2_id = int(name_to_id[p2_name])
        except KeyError as e:
            return f"Champion '{e.args[0]}' not found in mapping."

        try:
            stats = matchup_df.loc[(p1_id, p1_role, p2_id, p2_role, relationship)]
            win_rate_pct = f"{stats['win_rate'] * 100:.2f}%"
            return f"  - Win Rate: {win_rate_pct} ({int(stats['wins']):,} wins / {int(stats['total_games']):,} games)"
        except KeyError:
            return "  - Matchup not found in the data (likely filtered due to low play rate)."

    # 3. Perform the Spot-Checks
    print("\n--- Spot-Check 1: Classic Counter Matchup (Malphite vs. Yasuo) ---")
    print("Checking Malphite (TOP) vs. Yasuo (TOP):")
    print(get_matchup_stats('Malphite', 'TOP', 'Yasuo', 'TOP', 'enemy'))
    print("Checking Yasuo (TOP) vs. Malphite (TOP):")
    print(get_matchup_stats('Yasuo', 'TOP', 'Malphite', 'TOP', 'enemy'))
    
    print("\n--- Spot-Check 2: Synergistic Bot Lane (Lucian & Nami) ---")
    print("Checking Lucian (BOTTOM) with Nami (UTILITY):")
    print(get_matchup_stats('Lucian', 'BOTTOM', 'Nami', 'UTILITY', 'teammate'))
    print("Checking Nami (UTILITY) with Lucian (BOTTOM):")
    print(get_matchup_stats('Nami', 'UTILITY', 'Lucian', 'BOTTOM', 'teammate'))
    
    print("\n--- Spot-Check 3: Standard Mid Lane (Ahri vs. Zed) ---")
    print("Checking Ahri (MIDDLE) vs. Zed (MIDDLE):")
    print(get_matchup_stats('Ahri', 'MIDDLE', 'Zed', 'MIDDLE', 'enemy'))
    print("Checking Zed (MIDDLE) vs. Ahri (MIDDLE):")
    print(get_matchup_stats('Zed', 'MIDDLE', 'Ahri', 'MIDDLE', 'enemy'))
    
    print("\n--- Sanity check complete ---")

# Run the function
sanity_check_matchups()

--- Sanity-Checking 'safe/matchup_winrates.json' ---
Loading data files...
Data loaded and parsed successfully.
Found 391,442 total matchup entries.

--- Spot-Check 1: Classic Counter Matchup (Malphite vs. Yasuo) ---
Checking Malphite (TOP) vs. Yasuo (TOP):
  - Win Rate: 53.79% (9,091 wins / 16,900 games)
Checking Yasuo (TOP) vs. Malphite (TOP):
  - Win Rate: 46.19% (7,806 wins / 16,900 games)

--- Spot-Check 2: Synergistic Bot Lane (Lucian & Nami) ---
Checking Lucian (BOTTOM) with Nami (UTILITY):
  - Win Rate: 50.99% (392,920 wins / 770,554 games)
Checking Nami (UTILITY) with Lucian (BOTTOM):
  - Win Rate: 50.99% (392,920 wins / 770,554 games)

--- Spot-Check 3: Standard Mid Lane (Ahri vs. Zed) ---
Checking Ahri (MIDDLE) vs. Zed (MIDDLE):
  - Win Rate: 50.45% (65,003 wins / 128,838 games)
Checking Zed (MIDDLE) vs. Ahri (MIDDLE):
  - Win Rate: 49.54% (63,825 wins / 128,838 games)

--- Sanity check complete ---


In [9]:
import json
import os
import ast
from collections import defaultdict

# --- FILE CONFIGURATION ---
BASE_WR_FILE = "safe/base_winrates.json"
MATCHUP_WR_FILE = "safe/matchup_winrates.json"
VALID_ROLES_FILE = "safe/valid_champion_roles.json"
MAPPING_FILE = "safe/champion_mapping.json"
# ---

# --- DATA LOADING AND PRE-PROCESSING (should already be loaded in your notebook) ---
print("Loading and preparing all necessary data files (if not already loaded)...")

with open(MAPPING_FILE, 'r') as f:
    id_to_name = json.load(f)
name_to_id = {v: k for k, v in id_to_name.items()}

with open(VALID_ROLES_FILE, 'r') as f:
    valid_roles_list = json.load(f)
valid_roles_set = {(item['champion'], item['role']) for item in valid_roles_list}

with open(BASE_WR_FILE, 'r') as f:
    base_rates_list = json.load(f)
base_winrates = {(item['champion'], item['role']): item['win_rate'] for item in base_rates_list}

with open(MATCHUP_WR_FILE, 'r') as f:
    matchup_data_raw = json.load(f)

matchup_winrates = {}
for str_key, values in matchup_data_raw.items():
    key = ast.literal_eval(str_key)
    p1_id, p1_role, p2_id, p2_role, relationship = key
    p1_name = id_to_name.get(str(p1_id))
    p2_name = id_to_name.get(str(p2_id))
    
    if p1_name and p2_name:
        new_key = (p1_name, p1_role, p2_name, p2_role, relationship)
        matchup_winrates[new_key] = values['win_rate']

print("✅ All data loaded and ready.")
# --- END OF DATA LOADING ---


def get_recommendations(my_team, enemy_team, target_role):
    """
    Calculates the best champion picks for a target role based on the current draft.
    """
    recommendations = []
    unorthodox_picks = set()

    # 1. Validate the current draft (using the new structure)
    for role, champ in my_team.items():
        if champ and (champ, role) not in valid_roles_set: # Check only if a champion is actually picked
            unorthodox_picks.add(f"{champ} ({role})")
    for role, champ in enemy_team.items():
        if champ and (champ, role) not in valid_roles_set:
            unorthodox_picks.add(f"{champ} ({role})")
            
    # 2. Get the list of all champions we should consider for the target role
    champions_to_consider = [item['champion'] for item in valid_roles_list if item['role'] == target_role]
    
    # 3. Loop through every potential pick to calculate its score
    for pick_champ in champions_to_consider:
        pick_combo = (pick_champ, target_role)
        
        if pick_combo not in base_winrates:
            continue
            
        base_wr = base_winrates[pick_combo]
        total_delta = 0.0

        # Calculate delta against all picked enemies
        for enemy_role, enemy_champ in enemy_team.items():
            if enemy_champ and (enemy_champ, enemy_role) in valid_roles_set:
                matchup_key = (pick_champ, target_role, enemy_champ, enemy_role, 'enemy')
                matchup_wr = matchup_winrates.get(matchup_key, base_wr)
                total_delta += (matchup_wr - base_wr)

        # Calculate delta with all picked allies
        for ally_role, ally_champ in my_team.items():
            if ally_champ and (ally_champ, ally_role) in valid_roles_set:
                matchup_key = (pick_champ, target_role, ally_champ, ally_role, 'teammate')
                matchup_wr = matchup_winrates.get(matchup_key, base_wr)
                total_delta += (matchup_wr - base_wr)
        
        recommendations.append({
            "champion": pick_champ,
            "total_delta": total_delta,
            "base_win_rate": base_wr
        })
        
    # 4. Sort the recommendations
    recommendations.sort(key=lambda x: x['total_delta'], reverse=True)
    
    return recommendations, list(unorthodox_picks)


# --- EXAMPLE SIMULATION (with corrected structure) ---

# 1. Define the draft state using the new {"Role": "Champion"} format.
my_team = {
    "TOP": None,
    "JUNGLE": "Yuumi",
    "MIDDLE": None,
    "BOTTOM": "Mel",      # <--- This now clearly represents an open role
    "UTILITY": "Lulu"
}

enemy_team = {
    "TOP": None,
    "JUNGLE": "xDD",
    "MIDDLE": "Sylas",
    "BOTTOM": "Taric",
    "UTILITY": "Thresh"
}

# 2. Specify the role we want recommendations for
target_role_to_pick = "TOP"

# 3. Get the recommendations
recommendations, warnings = get_recommendations(my_team, enemy_team, target_role_to_pick)

# 4. Print the results
print("\n" + "="*50)
print(f"🏆 ALL PICKS FOR: {target_role_to_pick} (Sorted from best to worst) 🏆")
print("="*50)

if warnings:
    print(f"⚠️ WARNING: The following unorthodox picks were detected and ignored (delta set to 0):")
    for pick in warnings:
        print(f"  - {pick}")
    print("-"*50)

# Print ALL recommendations, not just the top 10
for i, pick in enumerate(recommendations):
    delta_str = f"+{pick['total_delta'] * 100:.2f}%" if pick['total_delta'] >= 0 else f"{pick['total_delta'] * 100:.2f}%"
    base_wr_str = f"{pick['base_win_rate'] * 100:.2f}%"
    
    print(f"{i+1:>3}. {pick['champion']:<12} | Total Delta: {delta_str:<8} | Base Win Rate: {base_wr_str}")

Loading and preparing all necessary data files (if not already loaded)...
✅ All data loaded and ready.

🏆 ALL PICKS FOR: TOP (Sorted from best to worst) 🏆
⚠️ WARNING: The following unorthodox picks were detected and ignored (delta set to 0):
  - Yuumi (JUNGLE)
  - Taric (BOTTOM)
  - xDD (JUNGLE)
--------------------------------------------------
  1. Velkoz       | Total Delta: +12.54%  | Base Win Rate: 51.36%
  2. Taric        | Total Delta: +10.97%  | Base Win Rate: 43.28%
  3. Vex          | Total Delta: +9.76%   | Base Win Rate: 49.89%
  4. Zoe          | Total Delta: +8.66%   | Base Win Rate: 50.87%
  5. RekSai       | Total Delta: +7.60%   | Base Win Rate: 51.50%
  6. Kassadin     | Total Delta: +6.61%   | Base Win Rate: 48.08%
  7. Kalista      | Total Delta: +6.33%   | Base Win Rate: 46.46%
  8. Diana        | Total Delta: +6.28%   | Base Win Rate: 47.59%
  9. Hwei         | Total Delta: +6.18%   | Base Win Rate: 51.03%
 10. Draven       | Total Delta: +5.70%   | Base Win Rate: